# SFT + GRPO on GSM8K (Qwen3-0.6B)

Before running:
- Set **Accelerator → GPU T4 x2** in the right panel
- Set **Internet → On**

Then click **Save Version → Save & Run All (Full Notebook)**. Safe to close your computer.

In [ ]:
# ── 1. Clone repo ─────────────────────────────────────────────────────────
import shutil, os, sys, subprocess

os.chdir('/kaggle/working')
shutil.rmtree('/kaggle/working/Fine-tuning-and-GRPO-on-Qwen', ignore_errors=True)
subprocess.run(['git', 'clone',
    'https://github.com/202520030411/Fine-tuning-and-GRPO-on-Qwen.git',
    '/kaggle/working/Fine-tuning-and-GRPO-on-Qwen'], check=True)

REPO = '/kaggle/working/Fine-tuning-and-GRPO-on-Qwen'
os.chdir(REPO)
sys.path.insert(0, REPO)
os.environ['HF_HOME']            = '/kaggle/working/.hf'
os.environ['HF_DATASETS_CACHE']  = '/kaggle/working/.hf/datasets'
os.environ['TRANSFORMERS_CACHE'] = '/kaggle/working/.hf/transformers'
print('Repo ready:', os.listdir('scripts'))

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', '-q',
    'transformers>=4.51.0', 'peft>=0.11.1', 'datasets>=2.19.0',
    'accelerate', 'tqdm', 'typer', 'sentencepiece', 'safetensors',
    'matplotlib', 'trl>=0.12.0', 'tiktoken'
], check=True)
print('Done')

In [ ]:
# ── 3. Download Qwen3-0.6B-Base ───────────────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer
AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B-Base')
AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-0.6B-Base')
print('Model cached')

In [ ]:
# ── 4. Prepare datasets (GSM8K, SVAMP, MMLU) ─────────────────────────────
!python scripts/prepare_gsm8k.py
!python scripts/prepare_svamp.py --limit 500
# MMLU is downloaded on-the-fly during eval (no prep script needed)

In [ ]:
# ── 5. Verify GPU ─────────────────────────────────────────────────────────
import torch
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}  {p.total_memory // 1024**2} MB')

In [ ]:
# ── 6. SFT training (~35 min on T4) ──────────────────────────────────────
!python scripts/train_sft.py \
    --model-name-or-path Qwen/Qwen3-0.6B-Base \
    --train-path dataset/processed/gsm8k_train.jsonl \
    --output-dir /kaggle/working/model/sft_gsm8k \
    --train-log-path /kaggle/working/model/sft_gsm8k/train_log.jsonl \
    --max-steps 300 \
    --per-device-batch-size 2 \
    --grad-accum 16 \
    --max-length 384 \
    --fp16 \
    --gradient-checkpointing \
    --log-every 10

In [ ]:
# ── 7. Eval base model ────────────────────────────────────────────────────
!python scripts/eval.py \
    --base-model Qwen/Qwen3-0.6B-Base \
    --test-path dataset/processed/gsm8k_test.jsonl \
    --output-path /kaggle/working/model/eval_base.jsonl \
    --max-new-tokens 256 \
    --batch-size 8 \
    --max-examples 500

In [ ]:
# ── 8. Eval SFT model ─────────────────────────────────────────────────────
!python scripts/eval.py \
    --base-model Qwen/Qwen3-0.6B-Base \
    --adapter-path /kaggle/working/model/sft_gsm8k \
    --test-path dataset/processed/gsm8k_test.jsonl \
    --output-path /kaggle/working/model/eval_sft.jsonl \
    --max-new-tokens 256 \
    --batch-size 8 \
    --max-examples 500

In [ ]:
# ── 9. GRPO training (~2.5 hrs on T4) ────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'trl>=0.12.0', 'tiktoken', 'peft>=0.11.1',
    'transformers>=4.51.0', 'accelerate'], check=True)

!python scripts/train_grpo.py \
    --model-name-or-path Qwen/Qwen3-0.6B-Base \
    --ref-model-path /kaggle/working/model/sft_gsm8k \
    --train-path dataset/processed/gsm8k_train.jsonl \
    --output-dir /kaggle/working/model/grpo_gsm8k \
    --max-steps 200 \
    --group-size 8 \
    --kl-coef 0.01

In [ ]:
# ── 10. Eval GRPO model ───────────────────────────────────────────────────
!python scripts/eval.py \
    --base-model Qwen/Qwen3-0.6B-Base \
    --adapter-path /kaggle/working/model/grpo_gsm8k \
    --test-path dataset/processed/gsm8k_test.jsonl \
    --output-path /kaggle/working/model/eval_grpo.jsonl \
    --max-new-tokens 256 \
    --batch-size 8 \
    --max-examples 500

In [ ]:
# ── 11. SVAMP eval (math generalisation) ─────────────────────────────────
!python scripts/eval.py \
    --base-model Qwen/Qwen3-0.6B-Base \
    --test-path dataset/processed/svamp_test.jsonl \
    --output-path /kaggle/working/model/eval_svamp_base.jsonl \
    --max-new-tokens 256 --batch-size 8 --max-examples 500

!python scripts/eval.py \
    --base-model Qwen/Qwen3-0.6B-Base \
    --adapter-path /kaggle/working/model/sft_gsm8k \
    --test-path dataset/processed/svamp_test.jsonl \
    --output-path /kaggle/working/model/eval_svamp_sft.jsonl \
    --max-new-tokens 256 --batch-size 8 --max-examples 500

!python scripts/eval.py \
    --base-model Qwen/Qwen3-0.6B-Base \
    --adapter-path /kaggle/working/model/grpo_gsm8k \
    --test-path dataset/processed/svamp_test.jsonl \
    --output-path /kaggle/working/model/eval_svamp_grpo.jsonl \
    --max-new-tokens 256 --batch-size 8 --max-examples 500

In [ ]:
# ── 12. MMLU eval (forgetting check) ─────────────────────────────────────
# subjects: high_school_mathematics, elementary_mathematics, world_history
MMLU_SUBJECTS="high_school_mathematics,elementary_mathematics,world_history"

!python scripts/eval_mmlu.py \
    --base-model Qwen/Qwen3-0.6B-Base \
    --output-path /kaggle/working/model/eval_mmlu_base.jsonl \
    --subjects "$MMLU_SUBJECTS" --max-examples-per-subject 150 --batch-size 8

!python scripts/eval_mmlu.py \
    --base-model Qwen/Qwen3-0.6B-Base \
    --adapter-path /kaggle/working/model/sft_gsm8k \
    --output-path /kaggle/working/model/eval_mmlu_sft.jsonl \
    --subjects "$MMLU_SUBJECTS" --max-examples-per-subject 150 --batch-size 8

!python scripts/eval_mmlu.py \
    --base-model Qwen/Qwen3-0.6B-Base \
    --adapter-path /kaggle/working/model/grpo_gsm8k \
    --output-path /kaggle/working/model/eval_mmlu_grpo.jsonl \
    --subjects "$MMLU_SUBJECTS" --max-examples-per-subject 150 --batch-size 8

In [ ]:
# ── 13. Analyze all results + plots ──────────────────────────────────────
!python scripts/analyze.py \
    --base-results  /kaggle/working/model/eval_base.jsonl \
    --sft-results   /kaggle/working/model/eval_sft.jsonl \
    --grpo-results  /kaggle/working/model/eval_grpo.jsonl \
    --svamp-base    /kaggle/working/model/eval_svamp_base.jsonl \
    --svamp-sft     /kaggle/working/model/eval_svamp_sft.jsonl \
    --svamp-grpo    /kaggle/working/model/eval_svamp_grpo.jsonl \
    --mmlu-base     /kaggle/working/model/eval_mmlu_base.jsonl \
    --mmlu-sft      /kaggle/working/model/eval_mmlu_sft.jsonl \
    --mmlu-grpo     /kaggle/working/model/eval_mmlu_grpo.jsonl \
    --sft-log       /kaggle/working/model/sft_gsm8k/train_log.jsonl \
    --grpo-log      /kaggle/working/model/grpo_gsm8k/trainer_state.json \
    --images-dir    /kaggle/working/images

In [ ]:
# ── 12. Zip everything for download ──────────────────────────────────────
import shutil
shutil.make_archive('/kaggle/working/results', 'zip', '/kaggle/working/model')
shutil.make_archive('/kaggle/working/plots',   'zip', '/kaggle/working/images')
print('ALL DONE — download results.zip and plots.zip from the Output panel')